In [20]:
import ast
import pandas as pd
import os
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np
import math
from src.metrics import *

In [21]:
mutant_dfs = []
datasets = ["GFP", "PHOT", "RASK", "YAP1"] #PHOT, RASK, YAP1, GFP

for dataset in datasets:
    path_to_data = f"../Data/Protein_Gym_Datasets/{dataset}.csv"
    mutant_df = pd.read_csv(path_to_data)
    mutant_dfs.append(mutant_df)


out_location = f"../../00_Uni/Thesis/Bilder/data_statistics"


/tmp/ipykernel_14821/385575638.py:6: DtypeWarning:

Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.



In [22]:
#Ratio of target mutants in the datasets
n_total_mutants = [len(mutant_df) for mutant_df in mutant_dfs]
n_target_mutants = []
scoring_column = "Norm_Score_1"
for mutant_df in mutant_dfs:
    target_score = mutant_df[scoring_column].max() * 0.95
    target_mutants = mutant_df[mutant_df[scoring_column] >= target_score]
    n_target_mutants.append(len(target_mutants))

print(datasets)
print( n_target_mutants)
print(n_total_mutants)
print([round(n_target/n_total,6) for n_target, n_total in zip(n_target_mutants, n_total_mutants)])

['GFP', 'PHOT', 'RASK', 'YAP1']
[80, 2, 3, 2]
[51714, 167511, 24873, 10075]
[0.001547, 1.2e-05, 0.000121, 0.000199]


In [23]:
#Statistics of the ddG filtered libraries:

library_dfs = [mutant_df[mutant_df["ΔΔG"] <= 0.5] for mutant_df in mutant_dfs]
n_target_mutants = []
scoring_column = "Norm_Score_1"
for mutant_df in library_dfs:
    target_score = mutant_df[scoring_column].max() * 0.95
    target_mutants = mutant_df[mutant_df[scoring_column] >= target_score]
    n_total_mutants = [len(library_df) for library_df in library_dfs]
    n_target_mutants.append(len(target_mutants))

print(datasets)
print( n_target_mutants)
print(n_total_mutants)
print([round(n_target/n_total,6) for n_target, n_total in zip(n_target_mutants, n_total_mutants)])

['GFP', 'PHOT', 'RASK', 'YAP1']
[78, 1, 3, 2]
[48756, 167372, 21775, 9735]
[0.0016, 6e-06, 0.000138, 0.000205]


In [5]:
"""Distribution of of number of mutations"""
# counts the total  number of mutations in the dataset and how many mutants have 1, 2, 3, ... mutations

all_mutation_counts = []
for mutants_df in mutant_dfs:
    mutation_counts = dict()
    for mutant in mutants_df["mutant"]:
        mutations = mutant.split(":")
        n_mutations = len(mutations)
        if n_mutations not in mutation_counts.keys():
            mutation_counts[n_mutations] = 1
        else:
            mutation_counts[n_mutations] += 1
    mutation_counts = dict(sorted(mutation_counts.items(), key=lambda x: x[0], reverse=False))
    all_mutation_counts.append(mutation_counts)


In [33]:
mutation_counts_plt = make_subplots(rows=1, cols=1)
colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA'] # Default Plotly colors
for i, mutation_counts in enumerate(all_mutation_counts):
    total_mutations = sum([mutation_counts[key] for key in mutation_counts.keys()])
    x_vals = [str(key) for key in all_mutation_counts[i].keys()]
    y_vals = [round(mutation_counts[key]/total_mutations, 2) for key in mutation_counts.keys()]

    mutation_counts_plt.append_trace(
    go.Bar(
        x=x_vals, 
        y=y_vals, 
        name=datasets[i],
        marker_color=colors[i % len(colors)],
        #text=[f"{y*100:.0f}%" for y in y_vals],
        textposition='outside',
        textfont=dict(color=colors[i % len(colors)], size=18)
        ), 
        row=1,col=1)

mutation_counts_plt.update_layout(
#title_text=f"Relative number of mutations and their frequency for all mutants",
#title_font=dict(color="black", size=24),
font=dict(color="black", size=18),
showlegend=True,
legend_title_text="Libraries",
paper_bgcolor='white',
plot_bgcolor='white',
width=2000,
height=800
)

mutation_counts_plt.update_layout(uniformtext_minsize=18, uniformtext_mode='show')

mutation_counts_plt.update_yaxes(dict(
title_text="Frequency of mutations(%)",
range=[0, 1],
color='black',
title_font=dict(color="black"),
tickfont=dict(color="black"),
showgrid=True,
linecolor='black',
gridcolor='grey',
griddash="dot",
gridwidth=0.5,
dtick = 0.1,
tickformat=".0%")
 )

mutation_counts_plt.update_xaxes(dict(
title_text="Number of Mutations per Sequence",
type="category",
color='black',
title_font=dict(color="black"),
tickfont=dict(color="black"),
gridcolor='grey',
griddash="dot",
tickson="boundaries",
showline=True,
linecolor='black',
zerolinecolor='black',
linewidth=1,
dtick=1,
showgrid=True,
),
) 

In [34]:
mutation_counts_plt.write_image(f"{out_location}/all_mutations_Counts.jpg")

In [24]:
"""Position_of_mutations"""
#create a dictionary where all positions of mutations and their frequency are listed

all_position_counts = []
for i, mutant_df in enumerate(mutant_dfs):
    positions_count = dict()
    for mutant in mutant_df["mutant"]:
        for mutation in mutant.split(":"):
            if mutant == 'WT' or mutant =='wAT' or mutant =='wt':
                continue

            try:
                position = int(mutation[1:-1])
            except Exception:
                print(mutant)
                continue

            if position not in positions_count.keys():
                positions_count.update({position: 1})
            else:
                positions_count.update({position: positions_count[position] + 1})

    positions_count = dict(sorted(positions_count.items(), key=lambda x: x[0], reverse=False))
    all_position_counts.append(positions_count)

In [25]:
positions_count_plt = make_subplots(
    rows=2, cols=2, 
    subplot_titles=(f'a) {datasets[0]}', f'b) {datasets[1]} (logarithmic)', f'c) {datasets[2]}', f'd) {datasets[3]}'),
    x_title="Position within Sequence",
    y_title="Number of Mutations"
)

for i, positions_count in enumerate(all_position_counts):
    for key in positions_count.keys():
        positions_count_plt.append_trace(
            go.Bar(
                x=[key], 
                y=[positions_count[key]]),
             row=1 if i in [1,2] else 2,
             col=1 if i in [0,2] else 2
             )

positions_count_plt.update_layout(
    showlegend=False,
    paper_bgcolor='white',
    plot_bgcolor='white',
    width=2000,
    height=1000,
    font=dict(color="black", size=22)
)

# Update subplot title font size
positions_count_plt.update_annotations(font=dict(color="black", size=22))

# Explicitly update the x_title and y_title annotations which might be overridden
for annotation in positions_count_plt['layout']['annotations']:
    if annotation['text'] in ["Position within Sequence", "Number of Mutations"]:
        annotation['font']['size'] = 22

# Set logarithmic scale for PHOT plot (row=1, col=2)
positions_count_plt.update_yaxes(
    type='log',
    range=[0,5.3],
    row=1, col=2
)

positions_count_plt.update_yaxes(dict(
    color='black',
    title_font=dict(color="black", size=22),
    tickfont=dict(color="black", size=18),
    showgrid=True,
    gridcolor='grey',
    griddash="dot",
    linecolor='black',
    gridwidth=0.5,
))

positions_count_plt.update_xaxes(dict(
    color='black',
    title_font=dict(color="black", size=22),
    tickfont=dict(color="black", size=18),
    showgrid=True,
    gridcolor='grey',
    griddash="dot",
    showline=True,
    zerolinecolor='black',
    linewidth=1,
    linecolor='black',
))

In [26]:
positions_count_plt.write_image(f"{out_location}/all_position_Counts.jpg")

In [28]:
# Bin the Norm_Score_1 values of the filtered dataframe
threshold = 1
score_bins = 20
bin_size = round(1/score_bins, 3)
# Define bin edges from 0 to 1 in steps of 0.05
bin_edges = np.arange(0, (1 + bin_size), bin_size)

scoring_distribution_plt = make_subplots(rows=1, cols=1)
colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA'] # Default Plotly colors
for i, mutants_df in enumerate(mutant_dfs):
    mutants_df['score_bin'] = pd.cut(mutants_df['Norm_Score_1'], bins=bin_edges, include_lowest=False) #labels=bin_labels,
    score_bin_counts = mutants_df["score_bin"].value_counts().sort_index()
    total_mutants = sum(score_bin_counts.values)
    
    x_vals = score_bin_counts.index.astype(str).tolist()
    y_vals = [round(score_bin_counts[key]/total_mutants,4) for key in score_bin_counts.keys()]

    scoring_distribution_plt.append_trace(
        go.Bar(
            x=x_vals, 
            y=y_vals,
            name=datasets[i],
            marker_color=colors[i % len(colors)],
            #text=[f"{y*100:.0f}%" if y > 0 else "" for y in y_vals],
            textposition='outside',
            textfont=dict(color=colors[i % len(colors)],
                            size=18)
            ),
            row=1, col=1
        )

scoring_distribution_plt.update_layout(
    font=dict(color="black", size=18),
    showlegend=True,
    legend_title_text="Libraries",
    paper_bgcolor='white',
    plot_bgcolor='white',
    width=2000,
    height=800
)

scoring_distribution_plt.update_layout(uniformtext_minsize=10, uniformtext_mode='show')



scoring_distribution_plt.update_yaxes(dict(
    title_text=f"relative frequency of mutants",
    color='black',
    title_font=dict(color="black"),
    tickfont=dict(color="black"),
    showgrid=True,
    gridcolor='grey',
    griddash="dot",
    tickformat=".0%",
    linecolor='black',
    gridwidth=0.5)
)

scoring_distribution_plt.update_xaxes(dict(
    title_text=f"Binned score of mutants",
    color='black',
    title_font=dict(color="black"),
    tickfont=dict(color="black"),
    tickangle=45,
    tickson="boundaries",
    zerolinecolor='black',
    linecolor='black',
    linewidth=1,
    dtick=1,
    showgrid=True,
    gridcolor='grey',
    showline=True,
    griddash="dot",

    ),
)

scoring_distribution_plt.show()

In [32]:
scoring_distribution_plt.write_image(f"{out_location}/all_scoring_distribution.jpg")

In [27]:
# Plotting Relationship between ZS Score (ΔΔG) and Normalized DMS Score with Thresholding

    
# Define threshold, for mutants to exclude from library
threshold = 0.5
p_correlations = [spearman_correlation(mutant_df["ΔΔG"], mutant_df["Norm_Score_1"]) for mutant_df in mutant_dfs]

subplot_titles = []
title_prefix = ['a', 'b', 'c', 'd']
for i in range(0,8):
    j = i if i < 4 else i - 4
    if i < 4:
        subplot_titles.append(f"{title_prefix[j]}) {datasets[j]} ρ = {p_correlations[j]:.2f}")
    else:
        subplot_titles.append(f"{title_prefix[j]}) {datasets[j]}")


zs_vs_score_comparions = make_subplots(
    cols=4,
    rows=2,
    subplot_titles=subplot_titles
) 
for i, mutant_df in enumerate(mutant_dfs):

    # Extract DMS and ZS scores
    zs_scores = mutant_df["ΔΔG"]
    dms_scores = mutant_df["Norm_Score_1"]

    # Group data based on threshold
    good_points = [(zs, dms) for zs, dms in zip(zs_scores, dms_scores) if zs <= threshold]
    bad_points = [(zs, dms) for zs, dms in zip(zs_scores, dms_scores) if zs > threshold]

    # DMS scores for boxplots
    good_dms = [dms for _, dms in good_points]
    bad_dms = [dms for _, dms in bad_points]

    # Compute means
    mean_good = sum(good_dms) / len(good_dms) if good_dms else 0
    mean_bad = sum(bad_dms) / len(bad_dms) if bad_dms else 0

    # Dotplots, first row
    zs_vs_score_comparions.add_trace(go.Scatter(
        x=[zs for zs, _ in good_points],
        y=[dms for _, dms in good_points],
        mode='markers',
        marker=dict(color='green', opacity=0.3),
        name=f'ZS ≤ {threshold}',
    ), row=1, col=i+1)

    zs_vs_score_comparions.add_trace(go.Scatter(
        x=[zs for zs, _ in bad_points],
        y=[dms for _, dms in bad_points],
        mode='markers',
        marker=dict(color='red', opacity=0.3),
        name=f'ZS > {threshold}',
    ), row=1, col=i+1)


    # Boxplots, second row
    zs_vs_score_comparions.add_trace(go.Box(
        y=bad_dms,
        boxmean=True,
        name=f'n={len(bad_dms)}',
        marker_color='red',
    ), row=2, col=i+1)

    zs_vs_score_comparions.add_trace(go.Box(
        y=good_dms,
        boxmean=True,
        name=f'n={len(good_dms)}',
        marker_color='green',
    ), row=2, col=i+1)


# Apply layout and axis updates outside the loop
zs_vs_score_comparions.update_layout(
    width=2000,
    height=900,
    showlegend=False,
    legend=dict(x=0.5, y=1.15, orientation='h', xanchor='center'),
    paper_bgcolor='white',
    plot_bgcolor='white',
    margin=dict(l=100, b=80) 
)

# X-axes for row 1
zs_vs_score_comparions.update_xaxes(
    tickfont=dict(color="black", size=18),
    showgrid=True,
    gridcolor='grey',

    griddash="dot",
    showline=True,
    zeroline=True,
    zerolinecolor='black',
    linecolor='black',
    zerolinewidth=1,
    row=1
)

# X-axes for row 2
zs_vs_score_comparions.update_xaxes(
    title_text="",
    tickfont=dict(color="black", size=18),
    showgrid=False,
    showline=True,
    linecolor='black',
    linewidth=1,
    zerolinecolor='black',
    row=2
)

zs_vs_score_comparions.update_yaxes(
    title_text="",
    tickfont=dict(color="black", size=18),
    showgrid=True,
    gridcolor='grey',
    griddash="dot",
    showline=True,
    linecolor='black',
    linewidth=1,
    zeroline=True,
    zerolinecolor='black',
    range=[0, None]
)

zs_vs_score_comparions.add_annotation(
    text="Normalized DMS Score",
    xref="paper", yref="paper",
    x=-0.04, y=0.5,
    xanchor="right",
    textangle=-90,
    showarrow=False,
    font=dict(size=22, color="black")
)

zs_vs_score_comparions.add_annotation(
    text="Change in Stability (kcal/mol)",
    xref="paper", yref="paper",
    x=0.5, y=0.58,
    yanchor="top",
    showarrow=False,
    font=dict(size=22, color="black")
)

zs_vs_score_comparions.add_annotation(
    text="Boxes and Counts for Excluded Mutants with ΔΔG > 0.5 kcal/mol (red) and Included Mutants with ΔΔG ≤ 0.5 kcal/mol (green)",
    xref="paper", yref="paper",
    x=0.5, y=-0.06,
    yanchor="top",
    showarrow=False,
    font=dict(size=22, color="black")
)

#zs_vs_score_comparions.show()
zs_vs_score_comparions.write_image(f"{out_location}/all_zs_vs_normScore.jpg")